# Visualization for the Cross VESRI meeting

Here I am reproducing the rotating 3d globe plot I previously made for the OM4 run with ZB20 vs 'regular' (not currently in this repo, somewhere in `EVENTS/2023_09_18_m2lines_site_visit_2023` on my LEAP hub storage).


## Required installs + functions

In [3]:
import xarray as xr
import numpy as np
import cftime
import cmocean as cm
import matplotlib.pyplot as plt
import regionmask
from xmip.regionmask import merged_mask
import cartopy.crs as ccrs
import os
import pandas as pd
from pandas import Timestamp
# from xarrayutils.plotting import linear_piecewise_scale

In [4]:
# Very long
Pred_path_temp = '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/2024-09-12_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHfTempOnly1975Epochs70Epoch55Years100_10repeat_36_6k_Train_global_3D_Test_global_3D_all_N_train_0_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_0_rand_seed_1.zarr' # temp no warming
Pred_path_all = '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/2024-09-12_ConvNextUNetTrain3Dv021Eval3DhfdsanomsNetZeroHf1975Epochs70Epoch55Years100_10repeat_36_6k_Train_global_3D_Test_global_3D_all_N_train_0_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_0_rand_seed_1.zarr' # all no warming

In [5]:
def post_processor(ds: xr.Dataset, ds_truth: xr.Dataset, ls) -> xr.Dataset:
    """Converts the prediction output to an xarray dataset with the same dimensions/variables as input"""
    # Always run the ds_input_validate in non-deep mode here

    # correct swapped dimensions and warn
    if len(ds.x) == 180 and len(ds.y) == 360:
        ds = ds.rename({"x": "x_i", "y": "y_i"}).rename({"x_i": "y", "y_i": "x"})


    da = ds["__xarray_dataarray_variable__"]
    n_lev = 19
    if set(ls) - {"zos"} == set(["uo", "vo", "thetao", "so"]):
        variables = ["uo", "vo", "thetao", "so"]
    elif set(ls) - {"zos"} == set(["thetao", "so"]):
        variables = ["thetao", "so"]
    elif set(ls) - {"zos"} == set(["uo", "vo"]):
        variables = ["uo", "vo"]
    slices = [slice(i, i + n_lev) for i in range(0, len(variables) * n_lev, n_lev)]
    var_slices = {k: sl for k, sl in zip(variables, slices)}
    variables = {
        k: da.isel(var=sl).rename({"var": "lev"}) for k, sl in var_slices.items()
    }
    variables["zos"] = da.isel(var=-1).squeeze()

    ds_out = xr.Dataset(variables)
    for var in ds_out.data_vars:
        if "lev" in ds_out[var].dims:
            ds_out[var] = ds_out[var].where(ds_truth.wetmask)
        else:
            ds_out[var] = ds_out[var].where(ds_truth.wetmask.isel(lev=0))

    ## attach all coordinates from input
    ds_out = ds_out.assign_coords({co: ds_truth[co] for co in ds_truth.coords})

    return ds_out

levels = 19
emulation_stability=True
smooth = False

# OM4 v0.2.1
ds_input = xr.open_zarr(
    os.path.join("/vast/sd5313/data/m2lines/3D_ocean_data/", "OM4_5daily_v0.2.1.zarr")
)

# Smooth the data 
if smooth:
    window = 10
    with ProgressBar():
        ds_input['uo'] = ds_input.uo.rolling(time=window, min_periods=1, center=False).mean().compute()
        ds_input['vo'] = ds_input.vo.rolling(time=window, min_periods=1, center=False).mean().compute()


# our groundtruth is always just a time slice of the training (training is a bad name

if emulation_stability:
    repeats = 3000
    ds_groundtruth = ds_input.isel(lev=slice(None, levels))
    ds_groundtruth = ds_groundtruth.sel(time=slice("1996-01-01", "1996-12-31"))
    new_time = np.arange(ds_groundtruth.time.size*repeats)
    ds_groundtruth = xr.concat([ds_groundtruth] * repeats, dim="time")
    ds_groundtruth['time'] = new_time
    ds_groundtruth = ds_groundtruth.isel(time=slice(3, 30000))

else:
    ds_groundtruth = ds_input.isel(time=slice(2903, 3503)).isel(lev=slice(None, levels))

ls_temp = ['thetao', 'so', 'zos']

output_folder_temp = Pred_path_temp.split("/")[-2].split("_Train")[0]
output_path_temp = os.path.join("./temp", output_folder_temp)


ds_prediction_raw_temp = xr.open_zarr(Pred_path_temp)
# if emulation_stability:
#     ds_groundtruth = ds_groundtruth.isel(time=slice(0, ds_prediction_all_raw.time.size))

# ds_prediction_all = post_processor(
#     ds_prediction_raw_all, ds_groundtruth, ls_all
# )


ds_prediction_temp = post_processor(
    ds_prediction_raw_temp, ds_groundtruth.isel(time = slice(0,ds_prediction_raw_temp.time.size)), ls_temp
)


# Run the test to make sure the output is formatted correctly
ds_prediction_temp = ds_prediction_temp.transpose('time','lev',...)

In [6]:
ds_prediction_temp['y']  = ds_prediction_temp.y.assign_attrs(long_name='latitude')
ds_prediction_temp['x']  = ds_prediction_temp.x.assign_attrs(long_name='longitude')
ds_prediction_temp['thetao'] = ds_prediction_temp['thetao'].assign_attrs(long_name = 'Temperature', units = r"${^oC}$")

In [7]:
#TODO: Factor these out into a module (WHAT IS THE EASIEST WAY TO DO THAT?) Do I need to build a full blown package?
import os

##### viz stuff (should also be factored out)
## camera stuff
def make_points(r,n, xoffset, yoffset, linear, funky, rotation_factor):
    """Helper to make XYZ points"""
    start = (2.55 + +0.25) * np.pi # the camera up gets confused when we are looking straight down. So start at a very slight angle
    rotation = (rotation_factor * np.pi)
    stop = start + rotation
    if linear:
        theta = np.linspace(start, stop, n)
    else:
        theta = np.geomspace(start, stop, n)
    
    if funky:
        # move closer and rotate a bit
        x = np.linspace(0, r/5, n)
        r = np.linspace(r, r*1/2, n)
    else:
        x = np.zeros(n)
    
    y = r * np.cos(theta)
    z = r * np.sin(theta)
    # swap dimensions
    z, x, y = x, y, z
    return np.column_stack((x+xoffset, y+yoffset, z))

def lines_from_points(points):
    """Given an array of points, make a line set"""
    poly = pv.PolyData()
    poly.points = points
    cells = np.full((len(points) - 1, 3), 2, dtype=np.int_)
    cells[:, 1] = np.arange(0, len(points) - 1, dtype=np.int_)
    cells[:, 2] = np.arange(1, len(points), dtype=np.int_)
    poly.lines = cells
    return poly


def orbital_path(r=1e-10,n=100, xoffset=0, yoffset=0, rotation_factor=1, linear=True, funky=False):
    points = make_points(r, n, xoffset, yoffset, linear, funky, rotation_factor)
    line = lines_from_points(points)
    line["scalars"] = np.arange(line.n_points)
    return line

def get_frame(ds, t, subsample, variable, nanmask, x_dim, y_dim, lon_name, lat_name):
    da = ds.isel({'time':t, x_dim:slice(None, None, subsample), y_dim:slice(None, None, subsample)}).drop('time')
    for co in [lon_name, lat_name]:
        da = da.assign_coords({co:da.coords[co].where(nanmask)})
    da = da[variable]
    return da

def make_mesh(da, lon_name, lat_name):
    lon = da[lon_name]
    lat = da[lat_name]
    mesh = gv.Transform.from_2d(lon, lat, data=da.data)
    # # Remove cells from the mesh with NaN values.
    mesh = mesh.threshold()
    return mesh

def update_mesh(plotter, ds, f, subsample, variable, clim, cmap, nanmask, x_dim, y_dim, lon_name, lat_name):
    da = get_frame(ds, f, subsample, variable, nanmask, x_dim, y_dim, lon_name, lat_name)
    mesh = make_mesh(da, lon_name, lat_name)
    sargs = {"title": f"{da.long_name} / {da.units}", 'color': 'White', 'bold':True}
    actor = plotter.add_mesh(
        mesh,
        show_edges=False,
        show_scalar_bar=True,
        scalar_bar_args=sargs,
        cmap=cmap,
        clim=clim,
        interpolate_before_map=True ######### 🔥 MIGHT AFFECT PERFORMANCE
    )
    return actor

def make_movie(ds_movie, variable, cmap, clim, filename, preview=False, x_dim='x', y_dim='y', lon_name='lon', lat_name='lat'):

    # some checks
    if any(len(ds_movie[co].dims)!=2 for co in [lon_name, lat_name]):
        raise ValueError(f"Expected longitude and latitude coordinates to be 2 dimensional. Got {[(co, list(ds_movie[co].dims)) for co in [lon_name, lat_name]]}")

    # It seems like some of the internal steps (`make_mesh` and in particular gv.Transform.from_2d(lon, lat, data=da.data) seem to assume that)
    # the dimensions are ordered in a certain way! 
    # TODO: Raise an issue and see if we can fix this upstream
    # for now always transpose here
    other_dims = [dim for dim in list(ds_movie.dims) if dim not in ['time', x_dim, y_dim]] #FIXME (also make time generic?)
    ds_movie = ds_movie.transpose(*['x', 'y', 'time']+other_dims)
    
    nanmask = ~np.isnan(ds_movie.isel(time=0)[variable]).reset_coords(drop=True)
    
    if preview:
        pv.global_theme.trame.interactive_ratio = 0.75 #1 or 2 
        pv.global_theme.trame.still_ratio = 4
        subs = 1
        window_size = None
    else:
        #(does this affect the movie quality?)
        pv.global_theme.trame.interactive_ratio = 5 #1 or 2 (does this affect the movie quality?)
        pv.global_theme.trame.still_ratio = 5
        subs = 1
        window_size = ([3840, 2160]) #4k res
        # window_size = ([1920, 1088]) #HD res
    
    
    p = gv.GeoPlotter(
        window_size= window_size,
        lighting='three lights'
    )
    
    # p.add_base_layer(texture=gv.natural_earth_1())
    # p.add_base_layer(texture=gv.natural_earth_hypsometric())
    p.add_coastlines()
    p.add_base_layer(color='gray')
    p.background_color = 'black'
    
    # first mesh add
    c = update_mesh(p, ds_movie, 0, subs, variable, clim, cmap, nanmask, x_dim, y_dim, lon_name, lat_name)

    ###### this whole block needs to be provided as an input class that just spits out camera position per frame
    move_fraction = 0.8
    n_time = len(ds_movie.time)
    frames = range(int(n_time*move_fraction))# make this 400
    no_move_frames = range(frames.stop, n_time)
    
    # create a camera path (isnt working because of the crs...
    opath = orbital_path(r=5, n=len(frames), funky=True, rotation_factor=0.9)
    p.camera_position = opath.points[0,:]
    p.camera.focal_point = (0, 0, 0)

    #### up until here

    movie_name = f'{filename}_{variable}.mp4'
    
    if preview:
        #$visualize the camera path
        tube = opath.tube(radius=0.006)
        p.add_mesh(tube, smooth_shading=True)
        p.camera_position = 'xz'
        p.show()
    else:
        p.open_movie(movie_name)
        p.write_frame()  # Write initial
        text_actor = None
        
        for f in tqdm(frames[1:]):
            # remove previous actor
            p.remove_actor(c)
            if text_actor:
                p.remove_actor(text_actor)
            # update actor
            c = update_mesh(p, ds_movie, f, subs, variable, clim, cmap, nanmask, x_dim, y_dim, lon_name, lat_name)
            # step camera
            p.camera.position = opath.points[f,:]
            p.camera.focal_point = (0, 0, 0)
            text_actor = p.add_text(
                    f"{str(ds_movie.isel(time=f).time.values)[:10]}",
                    position="lower_right",
                    font_size=30,
                    color="white",
                    shadow=False,
                )
            p.write_frame()  # Write this frame
            
            # p.camera.zoom(z)

        for f_still in tqdm(no_move_frames):
            # remove previous actor
            p.remove_actor(c)
            if text_actor:
                p.remove_actor(text_actor)
            # update actor
            c = update_mesh(p, ds_movie, f_still, subs, variable, clim, cmap, nanmask, x_dim, y_dim, lon_name, lat_name)
            text_actor = p.add_text(
                    f"{str(ds_movie.isel(time=f_still).time.values)[:10]}",
                    position="lower_right",
                    font_size=30,
                    color="white",
                    shadow=False,
                )
            p.write_frame()  # Write this frame
            
        print(f"Done with {movie_name}")
        # Be sure to close the plotter when finished
        p.close()
        

In [8]:
import xarray as xr

import pyvista as pv
# pl = pv.Plotter()
# print(pl.render_window.ReportCapabilities())

import geovista as gv
import geovista.theme
import pyvista as pv
import numpy as np
from tqdm.auto import tqdm
# pv.set_jupyter_backend("server") #https://github.com/pyvista/pyvista/issues/4652
# pv.global_theme.trame.server_proxy_enabled = True

In [9]:
gv.natural_earth_1()

Texture (0x14700fe056c0)
  Components:   3
  Cube Map:     False
  Dimensions:   7020, 3510

## Loading the data

>[!WARNING]
>While the docker image builds, lets forge ahead.

In [10]:
ds_prediction = ds_prediction_temp
def preprocess_movie(ds:xr.Dataset) -> xr.Dataset:
    # patch the 'seam' (TODO Move to the movie functions)
    ds_patch = ds.isel(x=0)
    ds_patch = ds_patch.assign_coords(x=360+ds_patch.x)
    ds = xr.concat([ds, ds_patch], dim='x')
    ds
    ##### just for the movie
    # add log10 of speed
    # speed = (ds.uo**2 + ds.vo**2)**0.5
    # ds['speed'] = np.log10(speed)
    
    
    # make 2d lon/lat arrays
    #??? Do we actually have lon/lat on the new 'clean' datasets?
    lon,lat = xr.broadcast(ds.x, ds.y)
    ds = ds.assign_coords(lon=lon, lat=lat)
    return ds

# ds_movie.isel(time=0).thetao.plot(x='lon', y='lat')
from dask.diagnostics import ProgressBar

with ProgressBar():
    ds_movie = preprocess_movie(ds_prediction.isel(time=slice(0,100)).isel(lev=0)).load()
    # ds_movie_depth = preprocess_movie(ds_prediction.isel(lev=5)).load()
    # ds_movie_groundtruth = preprocess_movie(ds_groundtruth.isel(lev=0)).load()

[########################################] | 100% Completed | 2.38 sms


### ATTENTION: THE NATURAL EARTH TEXTURE IS BROKEN

In [11]:
repeats = 10
dates = np.array(range(3, 365*100, 5))
new_time = np.array([np.datetime64('1990') + np.timedelta64(day-1,'D') for day in dates])
for i in range(1,repeats):
    new_time = np.hstack((new_time,np.array([np.datetime64(str(1996+i)) + np.timedelta64(day-1,'D') for day in dates])))
ds_movie['time'] = new_time[:len(ds_movie.time)]

/state/partition1/job-51054973/ipykernel_171307/4079079881.py:6: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds_movie['time'] = new_time[:len(ds_movie.time)]
/state/partition1/job-51054973/ipykernel_171307/4079079881.py:6: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constru

In [ ]:
cmap_dict={'thetao':'RdBu_r', 'zos':'RdBu', 'speed':'inferno', 'so':'cividis'}
# clim_dict={'tos':[6, 35], 'zos':[-1.5, 1.5], 'speed':[-1.5, 0.5]}
clim_dict={
    'thetao':[0,30],
    'so':[28, 39],
    'zos':[-1.5, 1.5],
    'speed':[-1., 0.7], # note the speed is already converted to log10!
}

v = 'thetao'
ds_movie['thetao'] = ds_movie['thetao'].assign_attrs(long_name = 'Potential Temperature', units = r"$( ^\circ C )$")
make_movie(ds_movie, v, cmap_dict[v], clim_dict[v], f"ocean_emulator_withfast", preview=False)

/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/pyvista/plotting/plotter.py:159: UserWarning: 
This system does not appear to be running an xserver.
PyVista will likely segfault when rendering.

Try starting a virtual frame buffer with xvfb, or using
  ``pyvista.start_xvfb()``

  warnings.warn(
